# Proyecto dental: segmentacion y rotulado con CNN

Notebook minimo para ejecutar el pipeline del proyecto y probar el modelo final. Usa los datasets guardados localmente en esta carpeta:

- `data/data/teeth_segmentation` para segmentacion/rotulado por pieza dental (`1` a `32`)
- `data/dental-disease-panoramic-detection-dataset`

El flujo se basa en los papers 2, 3 y 4: segmentacion con CNN, deteccion/rotulado de instancias y salida visual con mascaras, cajas y clases. El entrenamiento propio usa YOLO-seg sobre el dataset local segmentado de dientes cuando esta disponible. Para la demo final se usa un modelo YOLO-seg especializado en dientes con notacion FDI (`11` a `48`).

## Preparacion en Google Drive

Si subes esta carpeta a Google Drive, monta Drive y ajusta `PROJECT_PATH` a la ruta donde dejaste el proyecto. Mantiene el modelo FDI en `training_runs/dental_xray_cnn_segmentation/weights/best.pt` y guarda resultados nuevos en `outputs/`.


In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

# Cambia esta ruta si subiste la carpeta con otro nombre o dentro de otra carpeta.
PROJECT_PATH = "/content/drive/MyDrive/literatura"
os.chdir(PROJECT_PATH)
os.environ["PROJECT_DIR"] = f"{PROJECT_PATH}/outputs"

print("Carpeta actual:", os.getcwd())
print("Salidas en:", os.environ["PROJECT_DIR"])


In [ ]:
!pip -q install pyyaml opencv-python ultralytics


## Analisis y preparacion de datasets

Esta celda usa los datasets locales, analiza su estructura y prepara el YAML local sin entrenar todavia. Por defecto `TRAIN_DATASET=auto` prefiere `Teeth Segmentation` si existe.

In [ ]:
import os

os.environ["ANALYZE_ONLY"] = "1"
os.environ["TRAIN_DATASET"] = "auto"
!python dental_xray_pipeline.py


## Entrenamiento opcional

Entrena YOLOv8-seg como CNN de segmentacion de instancias usando el dataset local de dientes. Esta parte demuestra el entrenamiento del flujo con mascaras por pieza dental, pero sus clases son `1` a `32`, no FDI `11` a `48`. Si Colab da error de memoria, baja `BATCH` a `8`.

In [ ]:
import os

os.environ["ANALYZE_ONLY"] = "0"
os.environ["TRAIN_DATASET"] = "auto"  # auto usa Teeth Segmentation; disease usa hallazgos dentales
os.environ["EPOCHS"] = "15"
os.environ["IMGSZ"] = "512"
os.environ["BATCH"] = "16"
os.environ["WORKERS"] = "2"
os.environ["PATIENCE"] = "6"
os.environ["PREDICT_LIMIT"] = "24"
os.environ["CONF"] = "0.10"  # Baja a 0.05 si quieres mas detecciones, con mas falsos positivos

!python dental_xray_pipeline.py


## Inferencia sobre una radiografia con el modelo FDI

Esta celda usa el modelo FDI descargado (`best.pt`) para generar la vista final con dientes segmentados y rotulados. Si no existe, usa como respaldo el `best.pt` generado por el entrenamiento propio.


In [ ]:
from pathlib import Path

model_candidates = [
    Path(PROJECT_PATH) / "training_runs/dental_xray_cnn_segmentation/weights/best.pt",  # modelo FDI recomendado
    Path(PROJECT_PATH) / "outputs/training_runs/dental_xray_cnn_segmentation/weights/best.pt",  # modelo entrenado en Colab
]
MODEL_PATH = next((str(path) for path in model_candidates if path.exists()), None)
if MODEL_PATH is None:
    raise FileNotFoundError("No se encontro best.pt. Revisa training_runs/.../weights/best.pt o ejecuta el entrenamiento.")

default_test = Path(PROJECT_PATH) / "test.jpg"
IMAGE_PATH = str(default_test if default_test.exists() else Path("/content/mi_radiografia.jpg"))
OUTPUT_DIR = f"{PROJECT_PATH}/outputs/fdi_inference_result"

print("Modelo:", MODEL_PATH)
print("Imagen:", IMAGE_PATH)
!python predict_dental_xray.py --model "$MODEL_PATH" --image "$IMAGE_PATH" --output-dir "$OUTPUT_DIR" --imgsz 1280 --conf 0.25 --iou 0.50 --device cpu


## Salidas esperadas

- Modelo FDI recomendado: `$PROJECT_PATH/training_runs/dental_xray_cnn_segmentation/weights/best.pt`
- Imagen rotulada/segmentada final: `$PROJECT_PATH/outputs/fdi_inference_result/*_segmented_labeled.jpg`
- Reporte CSV final: `$PROJECT_PATH/outputs/fdi_inference_result/*_report.csv`
- Predicciones del entrenamiento propio: `$PROJECT_PATH/outputs/predictions_labeled_segmented/`
- Dataset YOLO-seg preparado desde Teeth Segmentation: `$PROJECT_PATH/data/_prepared_teeth_yolo_seg/`
- ZIP del entrenamiento propio: `$PROJECT_PATH/outputs/dental_xray_project_outputs.zip`